In [ ]:
!pip install --force-reinstall --no-cache-dir transformers==4.37.2
!pip install sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 62.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 177.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 267.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 300.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 372.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 259.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 241.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 367.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 362.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 367.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 318.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 295.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
Using cached huggingface_hub-1.6.0-py3-none-any.whl (612 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.37.2
    Uninstalling transformers-4.37.2:
      Successfully uninstalled transformers-4.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
doc1 = """
Retrieval Augmented Generation (RAG) is a technique that combines
information retrieval with large language models. Instead of relying
only on the model's training data, the system retrieves relevant
documents from a knowledge base before generating a response.
"""

doc2 = """
A vector database stores vector embeddings of text. These embeddings
capture semantic meaning and allow similarity search. FAISS is a
popular vector database used for efficient nearest neighbor search.
"""

documents = [doc1, doc2]

In [ ]:
def split_text(text, chunk_size=150):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = []
for doc in documents:
    chunks.extend(split_text(doc))

print(chunks)

['\nRetrieval Augmented Generation (RAG) is a technique that combines\ninformation retrieval with large language models. Instead of relying\nonly on the mo', "del's training data, the system retrieves relevant\ndocuments from a knowledge base before generating a response.\n", '\nA vector database stores vector embeddings of text. These embeddings\ncapture semantic meaning and allow similarity search. FAISS is a\npopular vector ', 'database used for efficient nearest neighbor search.\n']


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(4, 384)


In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 4


In [ ]:
query = "What is Retrieval Augmented Generation?"

In [ ]:
query_embedding = embedding_model.encode([query])

k = 2
distances, indices = index.search(np.array(query_embedding), k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved Context:\n")
for c in retrieved_chunks:
    print(c)

Retrieved Context:


Retrieval Augmented Generation (RAG) is a technique that combines
information retrieval with large language models. Instead of relying
only on the mo
del's training data, the system retrieves relevant
documents from a knowledge base before generating a response.



In [ ]:
context = " ".join(retrieved_chunks)

prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

result = generator(prompt, max_length=150)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the context below to answer the question.

Context:

Retrieval Augmented Generation (RAG) is a technique that combines
information retrieval with large language models. Instead of relying
only on the mo del's training data, the system retrieves relevant
documents from a knowledge base before generating a response.


Question:
What is Retrieval Augmented Generation?

Answer:
Retrieval Augmented Generation is a technique to combine
information retrieval with large language models. Instead of relying
only on the mo del's training data, the system retrieves relevant
documents from a knowledge base before generating a response.
Question:
What is Retrieval Augmented Generation?
Answer:
Retrieval Augmented Generation is a technique to combine
information retrieval with large language models. Instead of relying
only the mo del's training data, the system retrieves relevant
documents from a knowledge base before generating a response.
Question:
What is Retrieval Augmented Generation?
Answe